# Chapter 1 &mdash; Scanners and Parsers: Where This Theory Pays Off

**Concept 8 of the Chapter 1 decomposition:** *The Practical Payoff: Syntax Definition, Scanning, Parsing*

A scanner is "nothing but a highly simplified parser". We build a real one as a DFA and watch it turn text into tokens.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter1/Concept-Scanning-And-Parsing/Concept-Scanning-And-Parsing.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.AnimateDFA     import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


Once languages had formal syntax, programs had to be **parsed**. Compilers split the
job:

* the **scanner** (lexer) recognises identifiers, numbers, keywords &mdash; it is a
  **finite automaton**, and it produces *tokens*;
* the **parser** checks the overall shape of those tokens &mdash; it needs a **stack**.

`int My_var1, My_Var2;` becomes `keyword id , id ;`. This notebook builds the scanner
half as a DFA.

## 2. Definitions

### A scanner DFA for identifiers

An identifier is a letter followed by letters, digits or `_`.
We use `l` for "a letter", `d` for "a digit", `u` for underscore.

In [ ]:
ident = md2mc('''DFA
I  : l      -> F
I  : d | u  -> BH
F  : l | d | u -> F
BH : l | d | u -> BH
''')
print("identifier DFA states :", sorted(ident["Q"]))

### A scanner DFA for numbers

`-1.03` is legal; `30.-1` is not. Optional sign, digits, a point, then digits.

In [ ]:
# NOTE Jove convention: md2mc reads the state NAME. A name starting with
# 'I' is an initial state and one starting with 'F' is a final state, so
# intermediate states must avoid those letters. Calling these Iw/Ip would
# silently give the DFA four start states.
number = md2mc('''DFA
I      : s      -> Sgn
I      : d      -> Fwhole
I      : p      -> BH
Sgn    : d      -> Fwhole
Sgn    : p | s  -> BH
Fwhole : d      -> Fwhole
Fwhole : p      -> Dot
Fwhole : s      -> BH
Dot    : d      -> Ffrac
Dot    : p | s  -> BH
Ffrac  : d      -> Ffrac
Ffrac  : p | s  -> BH
BH     : d | p | s -> BH
''')
print("number DFA states :", sorted(number["Q"]))
print("start :", number["q0"], "  final :", sorted(number["F"]))
assert number["q0"] == "I", "state names starting with I are the START states"
assert number["F"] == {"Fwhole", "Ffrac"}, "names starting with F are the FINAL states"

### The tokeniser

Map real characters onto the DFA's alphabet, then ask the DFA.

In [ ]:
def classify(ch):
    if ch.isalpha(): return 'l'
    if ch.isdigit(): return 'd'
    if ch == '_':    return 'u'
    return None

def is_identifier(word):
    coded = ''.join(classify(c) or '?' for c in word)
    return '?' not in coded and accepts_dfa(ident, coded)

def encode_number(word):
    out = ''
    for ch in word:
        if   ch.isdigit(): out += 'd'
        elif ch == '.':    out += 'p'
        elif ch == '-':    out += 's'
        else:              return None
    return out

def is_number(word):
    e = encode_number(word)
    return e is not None and accepts_dfa(number, e)

## 3. Tests

The scanner must **accept** `Head1_ptr` and **reject** `1abc`.

In [ ]:
for w in ["Head1_ptr", "My_var1", "main", "x", "1abc", "_leading", ""]:
    print("%-12s identifier? %s" % (w, is_identifier(w)))
assert is_identifier("Head1_ptr") and not is_identifier("1abc")

And it must **reject `30.-1`** while **accepting `-1.03`** &mdash; the book's exact obligation.

In [ ]:
for w in ["-1.03", "1.0", "42", "30.-1", "1.", ".5", "-"]:
    print("%-10s number? %s" % (w, is_number(w)))
assert is_number("-1.03") and not is_number("30.-1")   # the book's exact obligation

Scanning a whole declaration into the token pattern a parser would see.

In [ ]:
def scan(line):
    toks = []
    for word in line.replace(',', ' , ').replace(';', ' ; ').split():
        if   word in (',', ';'):        toks.append(word)
        elif word in ('int', 'char'):   toks.append('keyword')
        elif is_number(word):           toks.append('number')
        elif is_identifier(word):       toks.append('id')
        else:                           toks.append('ERROR(%s)' % word)
    return ' '.join(toks)

print(scan("int My_var1, My_Var2, My_Var3;"))
print(scan("int My_var4 = -1.03;"))
print(scan("int 1bad;"))
assert scan("int My_var1, My_Var2, My_Var3;") == "keyword id , id , id ;"

## 4. Animation


Watch the identifier scanner consume a word, character by character.


*(The `display(HTML(...))` line loads the toolbar's font-awesome icons. Keep it last in the cell &mdash; it must be there for the controls to appear.)*

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(ident, FuseEdges=True)
display(HTML('<link rel="stylesheet" href="//stackpath.bootstrapcdn.com/font-awesome/4.7.0/css/font-awesome.min.css"/>'))

## 5. Exercises


1. Extend the identifier DFA so that `$` is also allowed after the first letter.
2. The number DFA rejects `1.` and `.5`. Should it? Change it so `1.` is accepted
   and say what that does to `30.-1`.
3. `scan` handles `=` as an error. Add it as its own token, then explain why the
   *parser* &mdash; not the scanner &mdash; must check that `keyword id = number ;` is well formed.

In [ ]:
# Your work for the exercises above.